# Figure S7

Draws representative WTD reconstructions with PI75 uncertainty bands.


In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS7'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy'
UNCERTAINTY_PATH = RECON / 'model_uncertainty' / 'monthly_model_uncertainty_radius_matrix.npy'
GRID_PATH = RECON / 'metadata' / 'grid_lookup.csv'
MONTH_PATH = RECON / 'metadata' / 'month_index.csv'
OBSERVATION_PATH = ROOT / 'data' / '2 well WTD' / '2011_2023' / 'wtd_monthly.csv'
FIGURE_PATH = OUT_DIR / 'FigS7b_uncertainty_examples_WTD.png'

FIXED_EXAMPLES = [
    {'example': 'Dense observations', 'grid_id': 45679},
    {'example': 'Moderate observations', 'grid_id': 75324},
    {'example': 'Sparse observations', 'grid_id': 56892},
]
X_AXIS_START = pd.Timestamp('2011-01-01')
X_AXIS_END = pd.Timestamp('2023-12-31')
X_AXIS_TICKS = pd.to_datetime([
    '2011-01-01', '2013-01-01', '2015-01-01', '2017-01-01',
    '2019-01-01', '2021-01-01', '2023-01-01',
])
WTD_Y_SPAN_M = 8.0

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.bf': 'Arial:bold',
    'axes.unicode_minus': False,
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'xtick.major.size': 3.0,
    'ytick.major.size': 3.0,
    'savefig.dpi': 900,
})


def display_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


print('Reconstruction:', display_path(RECON))
print('Output:', display_path(OUT_DIR))

In [ ]:
wtd_matrix = np.load(MATRIX_PATH, mmap_mode='r')
uncertainty_radius = np.load(UNCERTAINTY_PATH, mmap_mode='r')
grid = pd.read_csv(GRID_PATH)
month_index = pd.read_csv(MONTH_PATH)
month_index['date'] = pd.to_datetime(month_index['month_label'].astype(str) + '-01')

observations = pd.read_csv(
    OBSERVATION_PATH,
    usecols=['cell_id', 'month_label', 'mean_wtd_m'],
)
observations['date'] = pd.to_datetime(
    observations['month_label'].astype(str) + '-01'
)
observation_count = (
    observations.groupby('cell_id')['month_label']
    .nunique()
    .rename('n_obs_months')
)

if wtd_matrix.shape != uncertainty_radius.shape:
    raise ValueError(
        f'Reconstruction matrix {wtd_matrix.shape} and uncertainty matrix '
        f'{uncertainty_radius.shape} do not match.'
    )
if wtd_matrix.shape[0] != len(month_index):
    raise ValueError(
        f'Matrix months ({wtd_matrix.shape[0]}) do not match month index '
        f'({len(month_index)}).'
    )
if wtd_matrix.shape[1] != len(grid):
    raise ValueError(
        f'Matrix columns ({wtd_matrix.shape[1]}) do not match grid rows '
        f'({len(grid)}).'
    )

grid = grid.copy()
grid['matrix_col'] = np.arange(len(grid), dtype=int)
grid['mean_pi75_radius_m'] = np.nanmean(uncertainty_radius, axis=0)
grid = grid.merge(observation_count, on='cell_id', how='left')
grid['n_obs_months'] = grid['n_obs_months'].fillna(0).astype(int)

selected = pd.DataFrame(FIXED_EXAMPLES).merge(
    grid,
    on='grid_id',
    how='left',
    validate='one_to_one',
)
if selected['cell_id'].isna().any():
    missing = selected.loc[selected['cell_id'].isna(), 'grid_id'].astype(int).tolist()
    raise ValueError(f'Fixed grid_id values were not found in grid_lookup: {missing}')

selected[
    ['example', 'grid_id', 'cell_id', 'n_obs_months', 'mean_pi75_radius_m']
].round(3)

In [ ]:
series_color = '#3f5368'
band_color = '#9eb7cc'
observation_color = '#b45f35'
dates = month_index['date']

figure, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(7.2, 5.8),
    sharex=True,
    constrained_layout=True,
)
axes = np.asarray(axes).ravel().tolist()

for axis, (_, row) in zip(axes, selected.iterrows()):
    matrix_col = int(row['matrix_col'])
    cell_id = int(row['cell_id'])
    wtd = np.asarray(wtd_matrix[:, matrix_col], dtype=float)
    radius = np.asarray(uncertainty_radius[:, matrix_col], dtype=float)
    cell_observations = observations.loc[observations['cell_id'] == cell_id]

    axis.fill_between(
        dates,
        wtd - radius,
        wtd + radius,
        color=band_color,
        alpha=0.35,
        linewidth=0,
    )
    axis.plot(dates, wtd, color=series_color, linewidth=1.35)
    if not cell_observations.empty:
        axis.scatter(
            cell_observations['date'],
            cell_observations['mean_wtd_m'],
            s=10,
            color=observation_color,
            edgecolor='white',
            linewidth=0.25,
            zorder=4,
        )

    y_center = float(np.nanmedian(wtd))
    axis.set_ylim(
        y_center + WTD_Y_SPAN_M / 2,
        y_center - WTD_Y_SPAN_M / 2,
    )
    axis.yaxis.set_major_locator(mpl.ticker.MultipleLocator(2.0))
    axis.set_xlim(X_AXIS_START, X_AXIS_END)
    axis.set_xticks(X_AXIS_TICKS)
    axis.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axis.grid(True, color='#d8d8d8', linewidth=0.45, alpha=0.65)
    axis.tick_params(top=True, right=True, direction='out')
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
        spine.set_color('#303030')

for axis in axes:
    axis.set_ylabel('WTD (m)')
for axis in axes[:-1]:
    axis.set_xlabel('')
axes[-1].set_xlabel('Year')
axes[-1].tick_params(axis='x', labelrotation=35)

figure.savefig(FIGURE_PATH, bbox_inches='tight', dpi=900, facecolor='white')
plt.close(figure)

print('Saved:')
print('  ' + display_path(FIGURE_PATH))